# **HOMEWORK 1**

**Mateo Vivanco (00328476)** <br>
**Roberto Villafuerte (00329429)**

## Section 1: Research - Historical & Advanced Classical Ciphers

### The Vigenère Cipher:

**Mechanism:**

The Tabula Recta (Vigenère Square) is a 26×26 grid where each of the 26 rows is the standard alphabet shifted left by one additional position — row 0 is the normal alphabet (A–Z), row 1 shifts by 1 (B–Z, A), and so on. To encrypt, the keyword is repeated to match the length of the plaintext. For each plaintext letter, the corresponding keyword letter determines which row of the table to use; the plaintext letter then selects the column. The letter at that row–column intersection is the ciphertext.

**Example:** Keyword `KEY`, plaintext `HELLO`
- H + K (row 10) → R
- E + E (row 4) → I
- L + Y (row 24) → J
- L + K (row 10) → V
- O + E (row 4) → S  → Ciphertext: `RIJVS`

**Analysis (defense against frequency analysis):**
In a simple Caesar or monoalphabetic substitution cipher, each plaintext letter always maps to the same ciphertext letter, so the natural frequency distribution of a language (e.g., 'E' is most common in English) is preserved in the ciphertext and can be exploited. The Vigenère cipher is *polyalphabetic*: the same plaintext letter is encrypted differently depending on its position relative to the repeating keyword. A plaintext 'E' might become 'J' in one position and 'R' in another. This "flattens" the frequency distribution across the ciphertext, making simple frequency analysis ineffective.

**Breaking the Cipher — Kasiski Examination:**
1. **Find repeated sequences:** Scan the ciphertext for repeated trigrams or longer repeated substrings. Repetitions occur when identical sections of plaintext align with the same section of the repeated keyword.
2. **Record distances:** Note the distance (in characters) between each pair of repeated sequences.
3. **Compute GCD:** The keyword length is likely a divisor of each of those distances. Taking the GCD (or most common factor) of all the recorded distances gives a strong candidate for the keyword length $n$.
4. **Divide and conquer:** Once $n$ is known, the ciphertext is split into $n$ independent groups — each group is effectively a Caesar cipher and can be broken by standard frequency analysis.

### The Hill Cipher:

**Mathematical Basis:**
The Hill cipher operates on blocks of $n$ plaintext letters at a time.

1. **Vector conversion:** Each letter is assigned a numerical value (A=0, B=1, …, Z=25). A block of $n$ plaintext letters is arranged as a column vector $\mathbf{p} \in \mathbb{Z}_{26}^n$.
2. **Encryption:** The sender and receiver share an $n \times n$ key matrix $K$ with entries in $\mathbb{Z}_{26}$. The ciphertext block is computed as:
$$\mathbf{c} = K \cdot \mathbf{p} \pmod{26}$$
3. **Decryption:** The recipient recovers the plaintext via $\mathbf{p} = K^{-1} \cdot \mathbf{c} \pmod{26}$.

**Example (2×2 block):** Plaintext `HI` → vector $\mathbf{p} = \begin{pmatrix}7\\8\end{pmatrix}$. With key $K = \begin{pmatrix}3&3\\2&5\end{pmatrix}$: $\mathbf{c} = K\mathbf{p} \pmod{26} = \begin{pmatrix}45\\54\end{pmatrix} \pmod{26} = \begin{pmatrix}19\\2\end{pmatrix}$ → ciphertext `TC`.

**Requirements for the key matrix:**
The matrix $K$ must be **invertible modulo 26**, which requires two conditions:
1. $\det(K) \neq 0$ (non-zero determinant, so the matrix is algebraically invertible).
2. $\gcd(\det(K),\ 26) = 1$ — the determinant must be coprime to 26 (i.e., not divisible by 2 or 13), so that its modular inverse $(\det K)^{-1} \pmod{26}$ exists.

If either condition fails, $K^{-1} \pmod{26}$ does not exist and decryption is impossible.


### The Playfair Cipher:

**Grid construction:**
1. Write out the keyword, removing any duplicate letters.
2. Append the remaining letters of the alphabet (in order) that do not appear in the keyword. **I and J share a single cell** (the alphabet is reduced to 25 letters).
3. Fill the letters row-by-row into a $5 \times 5$ grid.

**Encryption rules (digraph substitution):**
First, split the plaintext into pairs of letters. If both letters in a pair are the same, insert a filler (usually **X**) between them and re-pair. If the message length is odd, append X at the end. Then apply:

| Condition | Rule |
|-----------|------|
| Same row | Replace each letter with the letter **immediately to its right** in the same row (wrap around). |
| Same column | Replace each letter with the letter **immediately below** it in the same column (wrap around). |
| Rectangle | Each letter is replaced by the letter in its **own row** but in the **other letter's column**. |

Decryption uses the inverse (left/up instead of right/down for the first two cases; the rectangle rule is self-inverse).

**Historical context — WWI and WWII:**
The Playfair cipher was invented by Charles Wheatstone in 1854 and championed by Lord Playfair. It was adopted by the British Army and saw extensive use from the Boer War through both World Wars. It was considered "field-ready" for several reasons:
- **No equipment required:** Operators needed only a pencil, paper, and a memorized keyword — no mechanical device or cipher machine.
- **Speed:** Trained soldiers could encrypt and decrypt messages quickly under field conditions.
- **Sufficient security:** It defeated casual interception and simple frequency analysis (since it operates on digraphs, not individual letters, it produces ~600 possible pairs versus 26 single letters, significantly complicating analysis).
- **Simplicity of training:** The five rules are easy to memorize and apply, unlike more complex book ciphers or machine systems that required specialist training.

### The Enigma Machine:

**Rotor Logic:**
Each rotor is a physical disc wired to perform a fixed letter-substitution (a permutation of the 26-letter alphabet). The Enigma typically used three (later four) rotors in series. When a key is pressed, an electrical signal passes through the rotors from right to left, through the reflector, and back left to right; the cumulative substitutions of all rotors determine the output (lit) letter.

Critically, **the rightmost rotor advances one step with every keystroke** — like the ones digit of an odometer. When it completes a full revolution (or reaches its notch position), it causes the middle rotor to step, and similarly for the left rotor. Because the rotor positions change, the effective substitution alphabet is **different for every single character encrypted**, producing a *polyalphabetic* cipher with an astronomically long period (e.g., for three rotors: $26^3 = 17{,}576$ steps before the pattern repeats, and with rotor selection the period is far longer).

**The Reflector:**
The reflector is a fixed (non-rotating) disc at the left end of the rotor stack that pairs up all 26 letters into 13 pairs and routes the signal back through the rotors in the reverse direction. Two consequences follow:
1. **Symmetric encryption/decryption:** Because the signal makes a return trip through the same rotors, the same machine settings encrypt and decrypt — sending operators did not need two different configurations.
2. **A letter can never encrypt as itself:** The reflector's wiring ensures that no letter is paired with itself. Combined with the symmetric return path, this means the ciphertext letter is always *different* from the plaintext letter. While convenient for operators, this was a fatal cryptographic weakness: Allied codebreakers at Bletchley Park exploited it to eliminate candidate settings (if a ciphertext letter matched the plaintext letter, that setting was impossible).

**The Plugboard (Steckerbrett):**
Before the signal entered the rotors (and again on the way out), it passed through the plugboard, where pairs of letters could be swapped using physical patch cables. In standard WWII operation, **10 cables** were used, swapping 10 pairs of letters (leaving 6 letters unchanged).

The number of ways to choose and pair 10 letters from 26 is:
$$\frac{26!}{(26-20)! \cdot 2^{10} \cdot 10!} = 150{,}738{,}274{,}937{,}250 \approx 1.5 \times 10^{14}$$

This factor of ~150 trillion was multiplied on top of the rotor settings, making the total keyspace approximately $10^{23}$ — far beyond any brute-force attack feasible at the time. The plugboard was therefore the single largest contributor to the Enigma's overall security.

## Section 2: Mathematical Foundations

### Modular Arithmetic

**a) $x \equiv 457 \pmod{13}$**

Divide 457 by 13:
$$457 = 35 \times 13 + 2 \quad \text{(since } 35 \times 13 = 455 \text{)}$$

$$\boxed{x \equiv 2 \pmod{13}}$$

**b) $x \equiv -25 \pmod{7}$**

Find the remainder of 25 divided by 7, then negate mod 7:
$$25 = 3 \times 7 + 4 \implies -25 \equiv -4 \equiv 7 - 4 \pmod{7}$$

Equivalently: $-25 = (-4) \times 7 + 3$

$$\boxed{x \equiv 3 \pmod{7}}$$

### The Euclidean Algorithm

**a) $\gcd(1160, 480)$**

Apply the algorithm repeatedly (divide, take remainder):

| Step | Equation |
|------|----------|
| 1 | $1160 = 2 \times 480 + 200$ |
| 2 | $480 = 2 \times 200 + 80$ |
| 3 | $200 = 2 \times 80 + 40$ |
| 4 | $80 = 2 \times 40 + 0$ |

The remainder is 0, so:

$$\boxed{\gcd(1160,\ 480) = 40}$$

### Extended Euclidean Algorithm — find $x, y$ such that $1160x + 480y = 40$

Work backwards through the steps above, expressing each remainder as a linear combination of the original numbers:

**Step 1 — express 40 from step 3:**
$$40 = 200 - 2 \times 80$$

**Step 2 — substitute 80 from step 2 ($80 = 480 - 2 \times 200$):**
$$40 = 200 - 2 \times (480 - 2 \times 200) = 200 - 2 \times 480 + 4 \times 200 = 5 \times 200 - 2 \times 480$$

**Step 3 — substitute 200 from step 1 ($200 = 1160 - 2 \times 480$):**
$$40 = 5 \times (1160 - 2 \times 480) - 2 \times 480 = 5 \times 1160 - 10 \times 480 - 2 \times 480$$

$$40 = 5 \times 1160 + (-12) \times 480$$

$$\boxed{x = 5, \quad y = -12}$$

**Verification:** $1160 \times 5 + 480 \times (-12) = 5800 - 5760 = 40$ 

### Euler's Totient Function

**Recall:** For $n = p_1^{a_1} p_2^{a_2} \cdots p_k^{a_k}$:
$$\phi(n) = n \prod_{p \mid n} \left(1 - \frac{1}{p}\right)$$

**a) $\phi(21)$**

$$21 = 3 \times 7 \qquad \text{(both prime)}$$

$$\phi(21) = (3-1)(7-1) = 2 \times 6 = \boxed{12}$$

**b) $\phi(17)$**

17 is prime, so:
$$\phi(17) = 17 - 1 = \boxed{16}$$

**c) $\phi(360)$**

Prime factorization:
$$360 = 2^3 \times 3^2 \times 5^1$$

Apply the formula:
$$\phi(360) = 360 \times \left(1 - \frac{1}{2}\right)\left(1 - \frac{1}{3}\right)\left(1 - \frac{1}{5}\right)$$

$$= 360 \times \frac{1}{2} \times \frac{2}{3} \times \frac{4}{5} = 360 \times \frac{8}{30} = \boxed{96}$$


### Shannon Entropy

The Shannon entropy of a discrete random variable $X$ is:
$$H(X) = -\sum_{i} P(x_i) \log_2 P(x_i)$$

| Symbol | $P(x_i)$ | $\log_2 P(x_i)$ | $-P(x_i)\log_2 P(x_i)$ |
|--------|-----------|-----------------|------------------------|
| A | $\frac{1}{2}$ | $-1$ | $\frac{1}{2}$ |
| B | $\frac{1}{4}$ | $-2$ | $\frac{1}{2}$ |
| C | $\frac{1}{8}$ | $-3$ | $\frac{3}{8}$ |
| D | $\frac{1}{8}$ | $-3$ | $\frac{3}{8}$ |

$$H(X) = \frac{1}{2} + \frac{1}{2} + \frac{3}{8} + \frac{3}{8} = 1 + \frac{6}{8} = \frac{7}{4}$$

$$\boxed{H(X) = 1.75 \text{ bits}}$$

Note: This result is optimal — it matches the bit lengths of an ideal Huffman code (A→`0`, B→`10`, C→`110`, D→`111`), confirming the source is a perfect binary tree distribution.


### Bayes' Theorem

**Given:**

| Symbol | Meaning | Value |
|--------|---------|-------|
| $P(A)$ | Probability of an actual attack | $0.001$ |
| $P(\bar{A})$ | Probability of no attack | $0.999$ |
| $P(F \mid A)$ | True Positive rate | $0.99$ |
| $P(F \mid \bar{A})$ | False Positive rate | $0.01$ |

**Find:** $P(A \mid F)$ — probability of a real attack given that the IDS flagged it.

**Step 1 — Total probability of a flag $P(F)$:**

$$P(F) = P(F \mid A)\,P(A) + P(F \mid \bar{A})\,P(\bar{A})$$
$$= (0.99)(0.001) + (0.01)(0.999) = 0.00099 + 0.00999 = 0.01098$$

**Step 2 — Apply Bayes' Theorem:**

$$P(A \mid F) = \frac{P(F \mid A)\,P(A)}{P(F)} = \frac{0.99 \times 0.001}{0.01098} = \frac{0.00099}{0.01098}$$

$$\boxed{P(A \mid F) \approx 0.0902 \approx 9.02\%}$$

**Interpretation:** Even though the IDS is quite accurate (99% true positive, 1% false positive), the base rate of attacks is very low ($P(A) = 0.001$). The vast majority of flagged events are false positives. This is a classic illustration of the **base rate fallacy** — when a rare event is tested with an imperfect detector, most positive results are false. In practice, a second verification step or additional context would be needed before responding to any single IDS alert.


## Section 3: PicoCTF

### Guess My Cheese

**Smart Approach:**

By what the scientists tells us, we know that we are working with a hashing instead of a cipher, and we have a list of cheese. By what the hints tells us, we can use the SHA256 hash function, and we need to append 2 hexadecimal values (0 - 255) to each cheese of the list, that we can then treat as a Rainbow Table.

In [1]:
import hashlib

cipher_text = '4d0c2b1732478d7b428f0386a91323e881bf9803e4387c9c1abdc00c58cbb09e'

cases = ['original', 'lowercase', 'uppercase']

cheeses = []

with open('cipher_texts/cheese_list.txt', 'r') as f:
    cheeses = [line.strip() for line in f]
    print(f'Loaded {len(cheeses)} cheeses from the list.')

for cheese in cheeses:
    for salt in range(256):
        for case in cases:
            if case == 'lowercase':
                cheese = cheese.lower()
            elif case == 'uppercase':
                cheese = cheese.upper()
        
            salted_cheese = cheese.encode('utf-8') + bytes([salt])
        
            hashed_cheese = hashlib.sha256(salted_cheese).hexdigest()
        
            if hashed_cheese == cipher_text:
                print(f'Found the cheese: {cheese} with salt: {format(salt, "02x")} and case: {case}')
                break

Loaded 599 cheeses from the list.
Found the cheese: texas goat cheese with salt: 2c and case: lowercase


Flag: **picoCTF{cHeEsY80eed518}**

**Rainbow Lookup Tables:**

Supposing we have no idea about any of the hints of the puzzle, since we are dealing with cheese, we can look for a list of cheeses online, and we can try to hash them with SHA256 (that we can determine by the length of the hashed cheese), and we can try to salt them in any way possible to try to find the secret cheese. Then we create a table of all the results and we try and look up out hash.

In [ ]:
import hashlib
import sys

target_hash = "4d0c2b1732478d7b428f0386a91323e881bf9803e4387c9c1abdc00c58cbb09e"

cases = ['original', 'lowercase', 'uppercase']
encodings = ['utf-8', 'latin-1', 'ascii', 'utf-16']
append_types = ['raw', 'hex']
append_positions = ['end', 'start', 'both']

cheeses = []

hash_table = {}

with open('cipher_texts/cheese_list.txt', 'r') as f:
    cheeses = [line.strip() for line in f]
    print(f'Loaded {len(cheeses)} cheeses from the list.')

for cheese in cheeses:
    for salt in range(256):
        for case in cases:
            for encoding in encodings:
                for append_type in append_types:
                    for append_position in append_positions:
                        modified_cheese = cheese
                        if case == 'lowercase':
                            modified_cheese = modified_cheese.lower()
                        elif case == 'uppercase':
                            modified_cheese = modified_cheese.upper()
                        
                        try:
                            encoded_cheese = modified_cheese.encode(encoding)
                        except UnicodeEncodeError:
                            continue
                        
                        if append_type == 'hex':
                            append_data = bytes([salt]).hex().encode('utf-8')
                        else:
                            append_data = bytes([salt])
                        
                        if append_position == 'end':
                            final_data = encoded_cheese + append_data
                        elif append_position == 'start':
                            final_data = append_data + encoded_cheese
                        else:  # both
                            final_data = append_data + encoded_cheese + append_data
                        
                        hashed_cheese = hashlib.sha256(final_data).hexdigest()
                        
                        hash_table[hashed_cheese] = (modified_cheese, salt, case, encoding)
                        

# Check if the target hash is in the hash table
if target_hash in hash_table:
    modified_cheese, salt, case, encoding = hash_table[target_hash]
    print(f'Found the cheese: {modified_cheese} with Salt: {format(salt, "02x")}, Case: {case}, Encoding: {encoding}')
else:
    print("Cheese not found in the list.")

Loaded 599 cheeses from the list.
Found the cheese: texas goat cheese with Salt: 2c, Case: lowercase, Encoding: ascii


Flag: **picoCTF{cHeEsY80eed518}**

### HideToSee

**Atbash Approach:**

Using a tool called "steghide" we can analyse the LSB of a .jpg file and extract hidden text. There we had found the encrypted flag, that we can decrypt using the same encryption method as the image itself suggests: Atbash Cipher.

In [9]:
cipher_text = 'krxlXGU{zgyzhs_xizxp_7142uwv9}'

alphabet = 'abcdefghijklmnopqrstuvwxyz'

plain_text = ''

for char in cipher_text:
    if char.lower() in alphabet:
        index = alphabet.index(char.lower())
        plain_text += alphabet[25 - index] if char.islower() else alphabet[25 - index].upper()
    else:
        plain_text += char

print(plain_text)

picoCTF{atbash_crack_7142fde9}


**Bruteforce Affine Cipher Approach:**

Again, using the same tool "steghide", we can find the cipher text. Then we can try to bruteforce the substitution cipher by assuming is an Affine Cipher for which we do not know the key, and by knowing that the flag has to start with "picoCTF".

In [ ]:
def find_mod_inverse(a, m):
    """Finds the modular multiplicative inverse of a under modulo m."""
    for i in range(1, m):
        if (a * i) % m == 1:
            return i
    return None

def decrypt_affine(ciphertext, a, b):
    """Decrypts text using the Affine formula: D(x) = a^-1 * (x - b) mod 26"""
    a_inv = find_mod_inverse(a, 26)
    if a_inv is None:
        return None

    plaintext = ""
    for char in ciphertext:
        if char.isalpha():
            # Convert to 0-25 format
            base = ord('A') if char.isupper() else ord('a')
            y = ord(char) - base
            
            # Apply the decryption formula
            x = (a_inv * (y - b)) % 26
            
            plaintext += chr(x + base)
        else:
            # Leave symbols (like CTF brackets) alone
            plaintext += char
            
    return plaintext

def brute_force_affine(ciphertext, known_flag_format="picoCTF"):
    print(f"Brute-forcing Affine Cipher for: {ciphertext}\n")
    
    # The 12 valid coprime multipliers for modulo 26
    valid_a_values = [1, 3, 5, 7, 9, 11, 15, 17, 19, 21, 23, 25]
    
    for a in valid_a_values:
        for b in range(26):
            decrypted = decrypt_affine(ciphertext, a, b)
            
            # If we know what we are looking for, catch it automatically
            if known_flag_format in decrypted:
                print("-" * 50)
                print(f"Found matching key.")
                print(f"Key: a = {a}, b = {b}")
                print(f"Plaintext: {decrypted}")
                print("-" * 50)
                return decrypted
                
    print("Brute force complete. Known flag format not found.")


cipher_text = "krxlXGU{zgyzhs_xizxp_7142uwv9}"
brute_force_affine(cipher_text)

Brute-forcing Affine Cipher for: krxlXGU{zgyzhs_xizxp_7142uwv9}

--------------------------------------------------
SUCCESS! Found matching key.
Key: a = 25, b = 25
Plaintext: picoCTF{atbash_crack_7142fde9}
--------------------------------------------------


'picoCTF{atbash_crack_7142fde9}'

### Rotation

**Brute Force Approach:**

We try all possible value rotations until we find one that starts with the string 'picoCTF'

In [26]:
cipher_text = 'xqkwKBN{z0bib1wv_l3kzgxb3l_429in00n}'

for i in range(26):
    decrypted = ''.join(
        chr((ord(c) - ord('a') - i) % 26 + ord('a')) if 'a' <= c <= 'z' else chr((ord(c) - ord('A') - i) % 26 + ord('A')) if 'A' <= c <= 'Z' else c
        for c in cipher_text
    )
    print(f"Shift {i}: {decrypted}")

    if "picoctf" in decrypted.lower():
        print(f"Possible flag found with shift {i}: {decrypted}")

Shift 0: xqkwKBN{z0bib1wv_l3kzgxb3l_429in00n}
Shift 1: wpjvJAM{y0aha1vu_k3jyfwa3k_429hm00m}
Shift 2: voiuIZL{x0zgz1ut_j3ixevz3j_429gl00l}
Shift 3: unhtHYK{w0yfy1ts_i3hwduy3i_429fk00k}
Shift 4: tmgsGXJ{v0xex1sr_h3gvctx3h_429ej00j}
Shift 5: slfrFWI{u0wdw1rq_g3fubsw3g_429di00i}
Shift 6: rkeqEVH{t0vcv1qp_f3etarv3f_429ch00h}
Shift 7: qjdpDUG{s0ubu1po_e3dszqu3e_429bg00g}
Shift 8: picoCTF{r0tat1on_d3crypt3d_429af00f}
Possible flag found with shift 8: picoCTF{r0tat1on_d3crypt3d_429af00f}
Shift 9: ohbnBSE{q0szs1nm_c3bqxos3c_429ze00e}
Shift 10: ngamARD{p0ryr1ml_b3apwnr3b_429yd00d}
Shift 11: mfzlZQC{o0qxq1lk_a3zovmq3a_429xc00c}
Shift 12: leykYPB{n0pwp1kj_z3ynulp3z_429wb00b}
Shift 13: kdxjXOA{m0ovo1ji_y3xmtko3y_429va00a}
Shift 14: jcwiWNZ{l0nun1ih_x3wlsjn3x_429uz00z}
Shift 15: ibvhVMY{k0mtm1hg_w3vkrim3w_429ty00y}
Shift 16: haugULX{j0lsl1gf_v3ujqhl3v_429sx00x}
Shift 17: gztfTKW{i0krk1fe_u3tipgk3u_429rw00w}
Shift 18: fyseSJV{h0jqj1ed_t3shofj3t_429qv00v}
Shift 19: exrdRIU{g0ipi1dc_s3rgnei3s_429pu00u}

**Extracting Rotation Value:**

Since we know that the flag has to start with 'picoCTF', we can extract the offset of the letters to finally find the whole flag.

In [10]:
cipher_text = 'xqkwKBN{z0bib1wv_l3kzgxb3l_429in00n}'

offset = ord('x') - ord('p')  # Calculate the offset based on the first character

decrypted_text = ''
for char in cipher_text:
    if 'a' <= char <= 'z':
        decrypted_char = chr((ord(char) - ord('a') - offset) % 26 + ord('a'))
    elif 'A' <= char <= 'Z':
        decrypted_char = chr((ord(char) - ord('A') - offset) % 26 + ord('A'))
    else:
        decrypted_char = char
    decrypted_text += decrypted_char

print(decrypted_text)

picoCTF{r0tat1on_d3crypt3d_429af00f}


### Substitution2

**English Pattern Recognition:**

We can manually assing some substitution values based on the fact that the flag has to start with 'picoCTF', so that:
- v -> p
- q -> i
- k -> c
- m -> o
- f -> t
- l -> f

And from that, we can guess some letters like n -> h and j -> e, to form the word 'THE' at the beginning. And so on guessing letters on pattern recognition of words in English.

In [ ]:
from collections import Counter

cipher_text = 'fnjdjjzqsfsjpjdxwmfnjdcjwwjsfxhwqsnjynqensknmmwkmuvafjdsjkadqftkmuvjfqfqmgsqgkwayqgekthjdvxfdqmfxgyaskthjdknxwwjgejfnjsjkmuvjfqfqmgslmkasvdquxdqwtmgstsfjusxyuqgqsfdxfqmglagyxujgfxwscnqknxdjpjdtasjlawxgyuxdojfxhwjsoqwwsnmcjpjdcjhjwqjpjfnjvdmvjdvadvmsjmlxnqensknmmwkmuvafjdsjkadqftkmuvjfqfqmgqsgmfmgwtfmfjxknpxwaxhwjsoqwwshafxwsmfmejfsfayjgfsqgfjdjsfjyqgxgyjzkqfjyxhmafkmuvafjdskqjgkjyjljgsqpjkmuvjfqfqmgsxdjmlfjgwxhmdqmasxllxqdsxgykmujymcgfmdaggqgeknjkowqsfsxgyjzjkafqgekmglqeskdqvfsmlljgsjmgfnjmfnjdnxgyqsnjxpqwtlmkasjymgjzvwmdxfqmgxgyquvdmpqsxfqmgxgymlfjgnxsjwjujgfsmlvwxtcjhjwqjpjxkmuvjfqfqmgfmaknqgemgfnjmlljgsqpjjwjujgfsmlkmuvafjdsjkadqftqsfnjdjlmdjxhjffjdpjnqkwjlmdfjknjpxgejwqsufmsfayjgfsqgxujdqkxgnqensknmmwsladfnjdcjhjwqjpjfnxfxgagyjdsfxgyqgemlmlljgsqpjfjkngqiajsqsjssjgfqxwlmdumagfqgexgjlljkfqpjyjljgsjxgyfnxffnjfmmwsxgykmglqeadxfqmglmkasjgkmagfjdjyqgyjljgsqpjkmuvjfqfqmgsymjsgmfwjxysfayjgfsfmogmcfnjqdjgjutxsjlljkfqpjwtxsfjxknqgefnjufmxkfqpjwtfnqgowqojxgxffxkojdvqkmkflqsxgmlljgsqpjwtmdqjgfjynqensknmmwkmuvafjdsjkadqftkmuvjfqfqmgfnxfsjjosfmejgjdxfjqgfjdjsfqgkmuvafjdskqjgkjxumgenqensknmmwjdsfjxknqgefnjujgmaenxhmafkmuvafjdsjkadqftfmvqiajfnjqdkadqmsqftumfqpxfqgefnjufmjzvwmdjmgfnjqdmcgxgyjgxhwqgefnjufmhjffjdyjljgyfnjqduxknqgjsfnjlwxeqsvqkmKFL{G6D4U_4G41T515_15_73Y10A5_42JX1770}'

key = {
    'v': 'p',
    'q': 'i',
    'k': 'c',
    'm': 'o',
    'k': 'c',
    'f': 't',
    'l': 'f',  

    # From now on, we will guess the characters based on a pattern analysis of the text (assuming it's English)
    'j': 'e',
    'f': 't',
    'n': 'h',
    'u': 'm',
    'g': 'n',
    's': 's',
    'a': 'u',
    'z': 'x',
    'd': 'r',
    't': 'y',
    'e': 'g',
    'w': 'l',
    'x': 'a',
    'h': 'b',
    'y': 'd',
    'p': 'v',
    'o': 'k',
    'c': 'w',
    'b': 'i',
}

plain_text = ''

for char in cipher_text:
    # Check if the character is in our key (handling lowercase for the main text)
    if char.lower() in key:
        plain_text += key[char.lower()].upper()
    else:
        # If we don't know it, leave it as the original character
        plain_text += char

print(plain_text)

print("\n" + "="*60)

# We print the flag found at the end of the text
import re
flag_match = re.search(r'PICOCTF\{.*?\}', plain_text)
if flag_match:
    print(f"Found Flag: {flag_match.group(0)}")

THEREEXISTSEVERALOTHERWELLESTABLISHEDHIGHSCHOOLCOMPUTERSECURITYCOMPETITIONSINCLUDINGCYBERPATRIOTANDUSCYBERCHALLENGETHESECOMPETITIONSFOCUSPRIMARILYONSYSTEMSADMINISTRATIONFUNDAMENTALSWHICHAREVERYUSEFULANDMARKETABLESKILLSHOWEVERWEBELIEVETHEPROPERPURPOSEOFAHIGHSCHOOLCOMPUTERSECURITYCOMPETITIONISNOTONLYTOTEACHVALUABLESKILLSBUTALSOTOGETSTUDENTSINTERESTEDINANDEXCITEDABOUTCOMPUTERSCIENCEDEFENSIVECOMPETITIONSAREOFTENLABORIOUSAFFAIRSANDCOMEDOWNTORUNNINGCHECKLISTSANDEXECUTINGCONFIGSCRIPTSOFFENSEONTHEOTHERHANDISHEAVILYFOCUSEDONEXPLORATIONANDIMPROVISATIONANDOFTENHASELEMENTSOFPLAYWEBELIEVEACOMPETITIONTOUCHINGONTHEOFFENSIVEELEMENTSOFCOMPUTERSECURITYISTHEREFOREABETTERVEHICLEFORTECHEVANGELISMTOSTUDENTSINAMERICANHIGHSCHOOLSFURTHERWEBELIEVETHATANUNDERSTANDINGOFOFFENSIVETECHNIiUESISESSENTIALFORMOUNTINGANEFFECTIVEDEFENSEANDTHATTHETOOLSANDCONFIGURATIONFOCUSENCOUNTEREDINDEFENSIVECOMPETITIONSDOESNOTLEADSTUDENTSTOKNOWTHEIRENEMYASEFFECTIVELYASTEACHINGTHEMTOACTIVELYTHINKLIKEANATTACKERPICOCTFISANOFFENSIVELYORIENT

**Frequency Analysis with MCMC:**

We start by assuming the text is written in English and that we are dealing with a substitution cipher. Then we train a Markov Chain Monte Carlo on the book: **The Adventures of Sherlock Holmes** (picked at random) and we generate a frequency model of English for different n-grams (specially bigrams), and then we use that training data to try and decrypt the text with a sort of random evolution process.

In [1]:
import urllib.request
import re
import math
import json
from collections import Counter

def build_bigram_model(url, output_filename="cipher_texts/english_bigrams.json"):
    print(f"Downloading training data from: {url}")
    
    try:
        # Fetch the book from Project Gutenberg
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req) as response:
            raw_text = response.read().decode('utf-8')
            
        print("Download complete. Cleaning text...")
        
        # Clean the text
        # Strip out all numbers, spaces, and punctuation. Convert to UPPERCASE.
        clean_text = re.sub(r'[^A-Z]', '', raw_text.upper())
        
        # Extract and count Bigrams
        print("Counting n-grams...")
        bigrams = [clean_text[i:i+2] for i in range(len(clean_text)-1)]
        bigram_counts = Counter(bigrams)
        
        total_bigrams = sum(bigram_counts.values())
        print(f"Processed {total_bigrams:,} total bigrams.")
        
        # Calculate Log-Probabilities
        print("Calculating log-probabilities...")
        bigram_log_probs = {}
        
        for bigram, count in bigram_counts.items():
            probability = count / total_bigrams # Probability = (Times this bigram appears) / (Total bigrams)
            
            bigram_log_probs[bigram] = round(math.log10(probability), 4) # Convert to log base 10 to prevent underflow errors in MCMC math
            
        # Save the model to a flat JSON dictionary
        with open(output_filename, 'w') as json_file:
            json.dump(bigram_log_probs, json_file, indent=4)
            
        print(f"Success! Bigram model saved to '{output_filename}'")
        
        # Show a sneak peek of the top 5 most common bigrams
        print("\n--- Top 5 Bigrams in Training Data ---")
        top_5 = bigram_counts.most_common(5)
        for bg, count in top_5:
            print(f"  '{bg}': {count:,} occurrences (Log-Prob: {bigram_log_probs[bg]})")
            
        return bigram_log_probs

    except Exception as e:
        print(f"Error generating model: {e}")

# URL for "The Adventures of Sherlock Holmes" by Arthur Conan Doyle (.txt format)
BOOK_URL = "https://www.gutenberg.org/cache/epub/1661/pg1661.txt"

# Run the generator
build_bigram_model(BOOK_URL)

Download complete. Cleaning text...
Counting n-grams...
Processed 446,756 total bigrams.
Calculating log-probabilities...
Success! Bigram model saved to 'cipher_texts/english_bigrams.json'

--- Top 5 Bigrams in Training Data ---
  'TH': 14,091 occurrences (Log-Prob: -1.5011)
  'HE': 12,181 occurrences (Log-Prob: -1.5644)
  'IN': 7,998 occurrences (Log-Prob: -1.7471)
  'ER': 7,950 occurrences (Log-Prob: -1.7497)
  'AN': 6,936 occurrences (Log-Prob: -1.809)


{'TH': -1.5011,
 'HE': -1.5644,
 'EP': -2.5879,
 'PR': -2.6728,
 'RO': -2.196,
 'OJ': -3.6372,
 'JE': -3.3623,
 'EC': -2.3299,
 'CT': -2.7212,
 'TG': -3.3759,
 'GU': -3.048,
 'UT': -2.3312,
 'TE': -2.1203,
 'EN': -1.9572,
 'NB': -3.0866,
 'BE': -2.2952,
 'ER': -1.7497,
 'RG': -3.0646,
 'GE': -2.5644,
 'EB': -2.6268,
 'BO': -2.8556,
 'OO': -2.4449,
 'OK': -2.9258,
 'KO': -3.5463,
 'OF': -2.1485,
 'FT': -2.5165,
 'EA': -1.9924,
 'AD': -2.3333,
 'DV': -3.4625,
 'VE': -2.0929,
 'NT': -2.0095,
 'TU': -2.7558,
 'UR': -2.3138,
 'RE': -1.875,
 'ES': -1.9203,
 'SO': -2.2731,
 'FS': -3.3645,
 'SH': -2.2466,
 'RL': -2.9845,
 'LO': -2.436,
 'OC': -2.7862,
 'CK': -2.763,
 'KH': -3.3972,
 'HO': -2.2257,
 'OL': -2.4778,
 'LM': -2.857,
 'ME': -2.1333,
 'ST': -1.9717,
 'HI': -1.9923,
 'IS': -1.9904,
 'SE': -2.1313,
 'KI': -2.9608,
 'SF': -2.9459,
 'FO': -2.415,
 'OR': -2.1014,
 'RT': -2.3955,
 'EU': -3.2155,
 'US': -2.4123,
 'EO': -2.4923,
 'FA': -2.5985,
 'AN': -1.809,
 'NY': -2.9027,
 'YO': -2.2427,


In [4]:
import json
import random
import math
import re

def load_ngram_model(filename="english_bigrams.json"):
    """Loads the pre-computed log-probabilities."""
    try:
        with open(filename, 'r') as f:
            return json.load(f)
    except FileNotFoundError:
        print(f"Could not find '{filename}'")
        return None

def score_text(text, bigram_model, default_penalty=-10.0):
    """Calculates fitness using standard dictionary lookups (no matrices)."""
    score = 0.0
    for i in range(len(text) - 1):
        bigram = text[i:i+2]
        score += bigram_model.get(bigram, default_penalty)
    return score

def mutate_key(current_key):
    """Swaps exactly two random letters in the decryption key."""
    new_key = current_key.copy()
    keys = list(new_key.keys())
    
    a, b = random.sample(keys, 2)
    new_key[a], new_key[b] = new_key[b], new_key[a]
    
    return new_key

def mcmc_decrypt(ciphertext, bigram_model, iterations=15000):
    print(f"Initializing MCMC Cracker with {iterations} iterations...\n")
    
    # Clean text to uppercase letters only
    clean_cipher = ''.join(c for c in ciphertext.upper() if c.isalpha())
    alphabet = list("ABCDEFGHIJKLMNOPQRSTUVWXYZ")
    
    # Start with a random key
    shuffled = alphabet.copy()
    random.shuffle(shuffled)
    current_key = {alphabet[i]: shuffled[i] for i in range(26)}
    
    # Apply initial key and score it
    current_decrypted = ''.join(current_key.get(c, c) for c in clean_cipher)
    current_score = score_text(current_decrypted, bigram_model)
    
    best_key = current_key
    best_score = current_score
    
    # The MCMC Loop
    for i in range(iterations):
        # Mutate and test
        proposed_key = mutate_key(current_key)
        proposed_decrypted = ''.join(proposed_key.get(c, c) for c in clean_cipher)
        proposed_score = score_text(proposed_decrypted, bigram_model)
        
        # Metropolis-Hastings Acceptance
        diff = proposed_score - current_score
        
        # Accept if better, or accept occasionally if worse to escape local peaks
        if diff > 0 or random.uniform(0, 1) < math.exp(diff):
            current_key = proposed_key
            current_score = proposed_score
            
            # Track the absolute best score globally
            if current_score > best_score:
                best_score = current_score
                best_key = current_key.copy()
                
        # Status update every 3000 iterations
        if i % 3000 == 0:
            preview = ''.join(best_key.get(c, c) for c in clean_cipher)[:60]
            print(f"Iteration {i:<6} | Score: {best_score:<8.2f} | Preview: {preview}...")

    print("\nMCMC Evolution Complete.")
    
    # Final Decryption Pass
    print("\n" + "="*60)
    print("FINAL DECRYPTED TEXT:")
    final_text = ''.join(best_key.get(c.upper(), c) for c in ciphertext)
    print(final_text)
    print("="*60 + "\n")

    # We print the flag found at the end of the text
    flag_match = re.search(r'PICOCTF\{.*?\}', final_text)
    if flag_match:
        print(f"Found Flag: {flag_match.group(0)}")
    
    return best_key


# Load the model we built from Sherlock Holmes
my_model = load_ngram_model("cipher_texts/english_bigrams.json")

target_cipher = "fnjdjjzqsfsjpjdxwmfnjdcjwwjsfxhwqsnjynqensknmmwkmuvafjdsjkadqftkmuvjfqfqmgsqgkwayqgekthjdvxfdqmfxgyaskthjdknxwwjgejfnjsjkmuvjfqfqmgslmkasvdquxdqwtmgstsfjusxyuqgqsfdxfqmglagyxujgfxwscnqknxdjpjdtasjlawxgyuxdojfxhwjsoqwwsnmcjpjdcjhjwqjpjfnjvdmvjdvadvmsjmlxnqensknmmwkmuvafjdsjkadqftkmuvjfqfqmgqsgmfmgwtfmfjxknpxwaxhwjsoqwwshafxwsmfmejfsfayjgfsqgfjdjsfjyqgxgyjzkqfjyxhmafkmuvafjdskqjgkjyjljgsqpjkmuvjfqfqmgsxdjmlfjgwxhmdqmasxllxqdsxgykmujymcgfmdaggqgeknjkowqsfsxgyjzjkafqgekmglqeskdqvfsmlljgsjmgfnjmfnjdnxgyqsnjxpqwtlmkasjymgjzvwmdxfqmgxgyquvdmpqsxfqmgxgymlfjgnxsjwjujgfsmlvwxtcjhjwqjpjxkmuvjfqfqmgfmaknqgemgfnjmlljgsqpjjwjujgfsmlkmuvafjdsjkadqftqsfnjdjlmdjxhjffjdpjnqkwjlmdfjknjpxgejwqsufmsfayjgfsqgxujdqkxgnqensknmmwsladfnjdcjhjwqjpjfnxfxgagyjdsfxgyqgemlmlljgsqpjfjkngqiajsqsjssjgfqxwlmdumagfqgexgjlljkfqpjyjljgsjxgyfnxffnjfmmwsxgykmglqeadxfqmglmkasjgkmagfjdjyqgyjljgsqpjkmuvjfqfqmgsymjsgmfwjxysfayjgfsfmogmcfnjqdjgjutxsjlljkfqpjwtxsfjxknqgefnjufmxkfqpjwtfnqgowqojxgxffxkojdvqkmkflqsxgmlljgsqpjwtmdqjgfjynqensknmmwkmuvafjdsjkadqftkmuvjfqfqmgfnxfsjjosfmejgjdxfjqgfjdjsfqgkmuvafjdskqjgkjxumgenqensknmmwjdsfjxknqgefnjujgmaenxhmafkmuvafjdsjkadqftfmvqiajfnjqdkadqmsqftumfqpxfqgefnjufmjzvwmdjmgfnjqdmcgxgyjgxhwqgefnjufmhjffjdyjljgyfnjqduxknqgjsfnjlwxeqsvqkmKFL{G6D4U_4G41T515_15_73Y10A5_42JX1770}"

if my_model:
    mcmc_decrypt(target_cipher, my_model)

Initializing MCMC Cracker with 15000 iterations...

Iteration 0      | Score: -5476.50 | Preview: QAIBIIPYCQCITIBLWEQAIBDIWWICQLMWYCAIFAYKACVAEEWVESZGQIBCIVGB...
Iteration 3000   | Score: -2988.89 | Preview: THEREEXISTSEVERALOTHERWELLESTABLISHEDHIGHSCHOOLCOMPUTERSECUR...
Iteration 6000   | Score: -2988.89 | Preview: THEREEXISTSEVERALOTHERWELLESTABLISHEDHIGHSCHOOLCOMPUTERSECUR...
Iteration 9000   | Score: -2988.89 | Preview: THEREEXISTSEVERALOTHERWELLESTABLISHEDHIGHSCHOOLCOMPUTERSECUR...
Iteration 12000  | Score: -2988.89 | Preview: THEREEXISTSEVERALOTHERWELLESTABLISHEDHIGHSCHOOLCOMPUTERSECUR...

MCMC Evolution Complete.

FINAL DECRYPTED TEXT:
THEREEXISTSEVERALOTHERWELLESTABLISHEDHIGHSCHOOLCOMPUTERSECURITYCOMPETITIONSINCLUDINGCYBERPATRIOTANDUSCYBERCHALLENGETHESECOMPETITIONSFOCUSPRIMARILYONSYSTEMSADMINISTRATIONFUNDAMENTALSWHICHAREVERYUSEFULANDMARKETABLESKILLSHOWEVERWEBELIEVETHEPROPERPURPOSEOFAHIGHSCHOOLCOMPUTERSECURITYCOMPETITIONISNOTONLYTOTEACHVALUABLESKILLSBUTALSOTOGETSTUDENTSINTERESTE

### Vigènere

**Reversed Encryption:**

Using the cipher word "CYLAB" we can reverse engineer the Vigenere Cipher to decrypt the flag.

In [65]:
# Vigenere Cipher Decryption
cipher_text = 'rgnoDVD{O0NU_WQ3_G1G3O3T3_A1AH3S_2951c89f}'

keyword = 'CYLAB'

encryption_table = []
for i in range(26):
    encryption_table.append([chr((j + i) % 26 + ord('A')) for j in range(26)])

print('Encrypted Text: ', cipher_text)

plain_text = ''

key_offset = 0

for i in range(len(cipher_text)):
    if cipher_text[i].isalpha():
        row = ord(keyword[(i - key_offset) % len(keyword)]) - ord('A')
        col = ord(cipher_text[i].upper()) - ord('A')
        plain_char = chr((col - row) % 26 + ord('A'))

        if cipher_text[i].islower():
            plain_char = plain_char.lower()
        plain_text += plain_char
    
    else:
        plain_text += cipher_text[i]
        key_offset += 1


print('Plain Text: ', plain_text)

Encrypted Text:  rgnoDVD{O0NU_WQ3_G1G3O3T3_A1AH3S_2951c89f}
Plain Text:  picoCTF{D0NT_US3_V1G3N3R3_C1PH3R_2951a89h}


**Keyword Extraction by Pattern:**

Knowing that the flag starts with "picoCTF", we can try and match it to the start of the cipher text and try to find the cipher word.

In [8]:
def vigenere_decrypt(cipher_text, keyword):
    plain_text = ''
    key_offset = 0

    for i in range(len(cipher_text)):
        if cipher_text[i].isalpha():
            row = ord(keyword[(i - key_offset) % len(keyword)].upper()) - ord('A')
            col = ord(cipher_text[i].upper()) - ord('A')
            plain_char = chr((col - row) % 26 + ord('A'))

            if cipher_text[i].islower():
                plain_char = plain_char.lower()
            plain_text += plain_char
        
        else:
            plain_text += cipher_text[i]
            key_offset += 1

    return plain_text

def find_key_by_pattern_matching(ciphertext):
    expected_start = "picoCTF"
    cipher_start = ""

    for char in ciphertext:
        if char.isalpha():
            cipher_start += char
        if len(cipher_start) >= len(expected_start):
            break

    print(f"Cipher start: {cipher_start}")
    print(f"Expected start: {expected_start}")

    key = ""
    for i in range(len(expected_start)):
        cipher_char = cipher_start[i].upper()
        plain_char = expected_start[i].upper()

        cipher_pos = ord(cipher_char) - ord('A')
        plain_pos = ord(plain_char) - ord('A')

        key_pos = (cipher_pos - plain_pos) % 26
        key_char = chr(key_pos + ord('A'))
        key += key_char

    print(f"Derived key from pattern: {key}")

    for key_length in range(1, len(key) + 1):
        test_key = key[:key_length]
        decrypted = vigenere_decrypt(ciphertext, test_key)

        print(f"\nTesting key '{test_key}':")
        print(f"- Decrypted: {decrypted}")

        if decrypted.startswith("picoCTF{") and decrypted.endswith("}"):
            print(f"Valid flag found with key: {test_key}")
            return decrypted, test_key

    return None, None

cipher_text = 'rgnoDVD{O0NU_WQ3_G1G3O3T3_A1AH3S_2951c89f}'
decrypted_text, found_key = find_key_by_pattern_matching(cipher_text)
if decrypted_text:
    print("\n" + "="*60)
    print(f"\nDecrypted flag: {decrypted_text}")
    print(f"Key used for decryption: {found_key}")

Cipher start: rgnoDVD
Expected start: picoCTF
Derived key from pattern: CYLABCY

Testing key 'C':
- Decrypted: pelmBTB{M0LS_UO3_E1E3M3R3_Y1YF3Q_2951a89d}

Testing key 'CY':
- Decrypted: pilqBXB{Q0LW_US3_E1I3M3V3_Y1CF3U_2951a89h}

Testing key 'CYL':
- Decrypted: picmFKB{Q0CS_YF3_E1I3D3R3_C1PF3U_2951r89d}

Testing key 'CYLA':
- Decrypted: picoBXS{O0LW_LQ3_E1I3D3T3_Y1CW3S_2951a89h}

Testing key 'CYLAB':
- Decrypted: picoCTF{D0NT_US3_V1G3N3R3_C1PH3R_2951a89h}
Valid flag found with key: CYLAB


Decrypted flag: picoCTF{D0NT_US3_V1G3N3R3_C1PH3R_2951a89h}
Key used for decryption: CYLAB


### picoCTF Flag Results

- Guess My Cheese (Part 2): picoCTF{cHeEsY80eed518}
- HideToSee: picoCTF{atbash_crack_7142fde9}
- Rotation: picoCTF{r0tat1on_d3crypt3d_429af00f}
- substitution2: picoCTF{N6R4M_4N41Y515_15_73D10U5_42EA1770}
- Vigenere: picoCTF{D0NT_US3_V1G3N3R3_C1PH3R_2951a89h}

picoCTF User: **ilovelacelili**

![PicoCTF](cipher_texts/PicoCTF.png)